In [1]:
import pandas as pd

---------FUNCTIONS-------------

In [2]:
#Function to convert an address to a ZIP code only
def codigo_postal(df):
    # 1. Retrieve the first 4 digits (CP4)
    # The [0] at the end is used to extract only the resulting text column
    cp4_orig = df['Endereço de recolha'].str.extract(r'(\d{4})')[0]
    cp4_dest = df['Endereço de destino'].str.extract(r'(\d{4})')[0]

    # 2. Enter the 4 digits in separate columns
    df['CP4_Origem'] = cp4_orig
    df['CP4_Destino'] = cp4_dest
    
    # 3. Try to enter the 3 digits following the hyphen (if any)
    cp3_orig = df['Endereço de recolha'].str.extract(r'\d{4}\s*-\s*(\d{3})')[0]
    cp3_dest = df['Endereço de destino'].str.extract(r'\d{4}\s*-\s*(\d{3})')[0]
    
    #4. THE MAGIC TRICK: Wherever cp3 is empty (NaN), we set it to '000'
    cp3_orig = cp3_orig.fillna('000')
    cp3_dest = cp3_dest.fillna('000')
    
    #5. Connect everything with a line to form CP7
    df['CP_Origem'] = cp4_orig + '-' + cp3_orig
    df['CP_Destino'] = cp4_dest + '-' + cp3_dest
    
    return df

In [3]:
def aplicar_rede_seguranca(df_viagens, df_dicionario):
    """
    A function that fills in empty locations using only the first 4 digits of the ZIP code.
    A bulletproof version that prevents name conflicts (_x and _y).
    """
    # 1. Prepare the generic dictionary and rename the column to avoid conflicts!
    df_dicionario['num_cod_postal'] = df_dicionario['num_cod_postal'].astype(str)
    df_gen = df_dicionario.drop_duplicates(subset=['num_cod_postal'])[['num_cod_postal', 'nome_localidade']]
    df_gen = df_gen.rename(columns={'nome_localidade': 'localidade_temp'}) # <--- HERE'S THE TRICK
    
    # 2. Internal mini-function
    def preencher_buracos(df, col_cp4, col_localidade):
        # We perform the merge
        df = pd.merge(df, df_gen, how='left', left_on=col_cp4, right_on='num_cod_postal')
        # We populate it with our temporary column
        df[col_localidade] = df[col_localidade].fillna(df['localidade_temp'])
        # We clean the house
        return df.drop(columns=['num_cod_postal', 'localidade_temp'])

    #3. Apply to the Source and the Destination
    df_viagens = preencher_buracos(df_viagens, 'CP4_Origem', 'nome_localidade_origem')
    df_viagens = preencher_buracos(df_viagens, 'CP4_Destino', 'nome_localidade_destino')
    
    return df_viagens

In [4]:
def adicionar_periodo_dia(df):
    # 1. Define the time slots and labels in English
    # 0-6: Night | 6-12: Morning | 12-16: Afternoon | 16-20: Rush hour | 20-24: Evening
    bins = [0, 6, 12, 16, 20, 24]
    labels = ['Night', 'Morning', 'Afternoon','Rush hour','Evening']
    
    # 2. Extract the time and create the new category
    # The ‘right=False’ parameter ensures that 12:00 is classified as Afternoon rather than Morning
    horas = df['Hora do pedido de viagem'].dt.hour
    df['Period_of_Day'] = pd.cut(horas, bins=bins, labels=labels, right=False, include_lowest=True)
    
    return df

---------CODE-------------

In [5]:
# Load the files into df (pandas library)
df_viagens = pd.read_excel('Viagens.xlsx',parse_dates=['Hora do pedido de viagem','Hora de chegada da viagem'])
df_pagamentos = pd.read_excel('Pagamentos.xlsx')
df_combustivel = pd.read_excel('Combustivel.xlsx', usecols=['DATA', 'TOTAL'])
df_CodigoPostal= pd.read_csv('CodigoPostal.csv')

In [6]:
# Calculating time efficiency:
df_viagens['Duracao'] = df_viagens['Hora de chegada da viagem'] - df_viagens['Hora do pedido de viagem']
# Pad potential errors with zeros before converting to an integer
df_viagens['Duracao_Minutos'] = (df_viagens['Duracao'].dt.total_seconds() / 60).fillna(0).round().astype(int)

In [7]:
#I create a new category with the day's height. I call the function `add_day_period`
df_viagens = adicionar_periodo_dia(df_viagens)

In [8]:
#Creation of the days of the week:
df_viagens['Week_Day'] = df_viagens['Hora do pedido de viagem'].dt.day_name()

In [9]:
# Filter to remove the `rider_cancelled` and `driver_cancelled` values
df_viagens = df_viagens[(df_viagens['Estado da viagem'] != 'rider_cancelled') & 
                        (df_viagens['Estado da viagem'] != 'driver_cancelled')]

In [10]:
df_viagens = df_viagens.rename(columns={'UUID da viagem': 'ID'})
df_pagamentos = df_pagamentos.rename(columns={'UUID da viagem': 'ID'})

In [11]:
# I selected the columns “Paid to you: Your income” and “Paid to you: Your income: Bonus.” These are the columns I will use from the df_pagamentos table.
colunas_selecionadas= ['Pago a si : Os seus rendimentos','Pago a si:Os seus rendimentos:Gratificação'] 
# Here, I've grouped the IDs from the `df_pagamentos` dataframe so they don't appear multiple times when merging the dataframes
df_pagamentos_agrupado = df_pagamentos.groupby('ID')[colunas_selecionadas].sum().reset_index()
# Join the `df_viagens` and `df_pagamentos` datasets
df_Def = pd.merge(
    df_viagens, 
    df_pagamentos_agrupado, 
    on='ID', 
    how='left'
)

In [12]:
#I renamed the imported columns so it wouldn't look so weird
df_Def=df_Def.rename(columns={
    'Pago a si : Os seus rendimentos': 'Rendimento',
    'Pago a si:Os seus rendimentos:Gratificação': 'Gorjeta'
})

In [13]:
#I call the function to convert the address to /zip code
df_Def = codigo_postal(df_Def)
df_Def.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1433 entries, 0 to 1432
Data columns (total 24 columns):
 #   Column                     Non-Null Count  Dtype          
---  ------                     --------------  -----          
 0   Source.Name                1433 non-null   object         
 1   ID                         1433 non-null   object         
 2   UUID do motorista          1433 non-null   object         
 3   Nome próprio do motorista  1433 non-null   object         
 4   Apelido do motorista       1433 non-null   object         
 5   UUID do veículo            1433 non-null   object         
 6   Matrícula                  1433 non-null   object         
 7   Tipo de serviço            1433 non-null   object         
 8   Hora do pedido de viagem   1433 non-null   datetime64[ns] 
 9   Hora de chegada da viagem  1433 non-null   datetime64[ns] 
 10  Endereço de recolha        1433 non-null   object         
 11  Endereço de destino        1433 non-null   object       

In [14]:
df_CodigoPostal.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 102358 entries, 0 to 102357
Data columns (total 4 columns):
 #   Column           Non-Null Count   Dtype 
---  ------           --------------   ----- 
 0   nome_localidade  102358 non-null  object
 1   num_cod_postal   102358 non-null  int64 
 2   ext_cod_postal   102358 non-null  int64 
 3   CP7              102358 non-null  object
dtypes: int64(2), object(2)
memory usage: 3.1+ MB


In [15]:
# Convert the columns in df_Def to integers
# We replace any empty values (NaN) with 0 and only then convert them to integers
df_Def['CP_Origem'] = df_Def['CP_Origem'].fillna(0).astype(str)
df_Def['CP_Destino'] = df_Def['CP_Destino'].fillna(0).astype(str)

In [16]:
#I remove the duplicates so there won't be any duplicates when I merge with df_Def
df_CodigoPostal = df_CodigoPostal.drop_duplicates(subset=['CP7'])

In [17]:
#I'm going to merge the data again using an Excel file that links the ZIP code to the city for the origin
df_Def = pd.merge(
    df_Def,
    df_CodigoPostal[['CP7','nome_localidade']],
    how='left',
    left_on= 'CP_Origem',
    right_on= 'CP7'
)
df_Def.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1433 entries, 0 to 1432
Data columns (total 26 columns):
 #   Column                     Non-Null Count  Dtype          
---  ------                     --------------  -----          
 0   Source.Name                1433 non-null   object         
 1   ID                         1433 non-null   object         
 2   UUID do motorista          1433 non-null   object         
 3   Nome próprio do motorista  1433 non-null   object         
 4   Apelido do motorista       1433 non-null   object         
 5   UUID do veículo            1433 non-null   object         
 6   Matrícula                  1433 non-null   object         
 7   Tipo de serviço            1433 non-null   object         
 8   Hora do pedido de viagem   1433 non-null   datetime64[ns] 
 9   Hora de chegada da viagem  1433 non-null   datetime64[ns] 
 10  Endereço de recolha        1433 non-null   object         
 11  Endereço de destino        1433 non-null   object       

In [18]:
df_Def=df_Def.rename(columns={'CP7':'CP7_Origem','nome_localidade':'nome_localidade_origem'})

In [19]:
# I'm going to merge the data again using an Excel file that links the ZIP code to the city, but this time for the destination
df_Def = pd.merge(
    df_Def,
    df_CodigoPostal[['CP7','nome_localidade']],
    how='left',
    left_on= 'CP_Destino',
    right_on= 'CP7'
)
df_Def.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1433 entries, 0 to 1432
Data columns (total 28 columns):
 #   Column                     Non-Null Count  Dtype          
---  ------                     --------------  -----          
 0   Source.Name                1433 non-null   object         
 1   ID                         1433 non-null   object         
 2   UUID do motorista          1433 non-null   object         
 3   Nome próprio do motorista  1433 non-null   object         
 4   Apelido do motorista       1433 non-null   object         
 5   UUID do veículo            1433 non-null   object         
 6   Matrícula                  1433 non-null   object         
 7   Tipo de serviço            1433 non-null   object         
 8   Hora do pedido de viagem   1433 non-null   datetime64[ns] 
 9   Hora de chegada da viagem  1433 non-null   datetime64[ns] 
 10  Endereço de recolha        1433 non-null   object         
 11  Endereço de destino        1433 non-null   object       

In [20]:
# Rename the newly created columns
df_Def=df_Def.rename(columns={'CP7':'CP7_Destino','nome_localidade':'nome_localidade_destino'})

In [21]:
# I'm deleting columns that no longer serve a purpose now that new ones have appeared
df_Def = df_Def.drop(columns=['Endereço de recolha','Endereço de destino'])

In [22]:
df_Def.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1433 entries, 0 to 1432
Data columns (total 26 columns):
 #   Column                     Non-Null Count  Dtype          
---  ------                     --------------  -----          
 0   Source.Name                1433 non-null   object         
 1   ID                         1433 non-null   object         
 2   UUID do motorista          1433 non-null   object         
 3   Nome próprio do motorista  1433 non-null   object         
 4   Apelido do motorista       1433 non-null   object         
 5   UUID do veículo            1433 non-null   object         
 6   Matrícula                  1433 non-null   object         
 7   Tipo de serviço            1433 non-null   object         
 8   Hora do pedido de viagem   1433 non-null   datetime64[ns] 
 9   Hora de chegada da viagem  1433 non-null   datetime64[ns] 
 10  Distância da viagem        1430 non-null   float64        
 11  Estado da viagem           1433 non-null   object       

In [23]:
# I mapped the function `aplicar_rede_segurança` to assign values to ZIP codes that contain only the first 4 digits
df_Def = aplicar_rede_seguranca(df_Def, df_CodigoPostal)
df_Def.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1433 entries, 0 to 1432
Data columns (total 26 columns):
 #   Column                     Non-Null Count  Dtype          
---  ------                     --------------  -----          
 0   Source.Name                1433 non-null   object         
 1   ID                         1433 non-null   object         
 2   UUID do motorista          1433 non-null   object         
 3   Nome próprio do motorista  1433 non-null   object         
 4   Apelido do motorista       1433 non-null   object         
 5   UUID do veículo            1433 non-null   object         
 6   Matrícula                  1433 non-null   object         
 7   Tipo de serviço            1433 non-null   object         
 8   Hora do pedido de viagem   1433 non-null   datetime64[ns] 
 9   Hora de chegada da viagem  1433 non-null   datetime64[ns] 
 10  Distância da viagem        1430 non-null   float64        
 11  Estado da viagem           1433 non-null   object       

In [24]:

# Calculate the cost per kilometer
total_combustivel = df_combustivel['TOTAL'].sum()
total_km = df_Def['Distância da viagem'].sum()
custo_por_km = total_combustivel / total_km


# Apply directly to df_Def
df_Def['Custo_Combustivel'] = df_Def['Distância da viagem'] * custo_por_km
df_Def['Lucro_Liquido'] = df_Def['Rendimento'] - df_Def['Custo_Combustivel']

In [25]:
# I'm going to delete the columns I created that I don't think make much sense to keep 
df_Def['Data'] = df_Def['Hora do pedido de viagem'].dt.date

In [26]:
#Suspected duplicates after the merge, as we can see 
duplicados=df_Def[df_Def.duplicated(subset=['ID'], keep=False)]
id_suspeito = duplicados['ID'].iloc[50]
df_Def[df_Def['ID'] == id_suspeito]
#After confirmation, I deleted the values
df_Def = df_Def.drop_duplicates(subset=['ID'], keep='first')

In [27]:
#I deleted the null values in the “Income” column because there were only two of them
df_Def = df_Def.dropna(subset=['Rendimento'])

In [28]:
#Values I couldn't identify. Most likely outside the port or in Gaia. I decided to list them under “Others.”
df_Def.loc[:, 'nome_localidade_origem'] = df_Def['nome_localidade_origem'].fillna('Others')
df_Def.loc[:, 'nome_localidade_destino'] = df_Def['nome_localidade_destino'].fillna('Others')
df_Def = df_Def.drop(columns=['Hora do pedido de viagem','Hora de chegada da viagem','CP4_Origem','CP4_Destino','CP_Origem','CP_Destino'])

In [29]:
#I exported the document to Excel so I could work on it in another worksheet
df_Def.to_excel("Dados_Limpos.xlsx")

In [30]:
df_Def.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1278 entries, 0 to 1432
Data columns (total 23 columns):
 #   Column                     Non-Null Count  Dtype          
---  ------                     --------------  -----          
 0   Source.Name                1278 non-null   object         
 1   ID                         1278 non-null   object         
 2   UUID do motorista          1278 non-null   object         
 3   Nome próprio do motorista  1278 non-null   object         
 4   Apelido do motorista       1278 non-null   object         
 5   UUID do veículo            1278 non-null   object         
 6   Matrícula                  1278 non-null   object         
 7   Tipo de serviço            1278 non-null   object         
 8   Distância da viagem        1278 non-null   float64        
 9   Estado da viagem           1278 non-null   object         
 10  Duracao                    1278 non-null   timedelta64[ns]
 11  Duracao_Minutos            1278 non-null   int64          
 1